In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
module_path = os.path.abspath(os.path.join('../../../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
print(sys.path)

['/usr/lib/python311.zip', '/usr/lib/python3.11', '/usr/lib/python3.11/lib-dynload', '', '/data/isa/mlex/venv/lib/python3.11/site-packages', '/data/isa/mlex']


In [4]:
import pandas as pd

In [5]:
base_path = r'/data/eeg_eyestate/EEG_Eye_State.csv'
label_column = 'eyeDetection'
df = pd.read_csv(base_path, delimiter=';')


In [ ]:
df

In [6]:
counts = df[label_column].value_counts()
        
# 4. Calcular a porcentagem
total = counts.sum()
percentages = (counts / total) * 100

# 5. Imprimir os resultados de forma clara
print("\n--- Análise de Classes (eyedetection) ---")
print(f"Total de amostras: {total:,}".replace(",", "."))
print("-" * 35)

for label, count in counts.items():
    percent = percentages.loc[label]
    print(f"Classe {label}: {count:,} amostras ({percent:.2f}%)".replace(",", "."))





--- Análise de Classes (eyedetection) ---
Total de amostras: 11.572
-----------------------------------
Classe 1: 6.033 amostras (52.13%)
Classe 0: 5.539 amostras (47.87%)


In [7]:
df_analise = pd.DataFrame({
    'Count': counts,
    'Percentage': percentages
})

df_analise.index = df_analise.index.map({
    0: 'Eyes Open (0)',
    1: 'Eyes Closed (1)'
})

# --------------------------------------------------------------------------
# --- FUNÇÃO CORRIGIDA PARA REMOVER TODO O LIXO DO PANDAS TO_LATEX ---
# --------------------------------------------------------------------------

def gerar_latex_analise_classes(df):
    
    # 3.1. Criar um DataFrame formatado com as strings finais
    df_latex = pd.DataFrame({
        'Samples': df['Count'].apply(lambda x: f'{x:,}'.replace(',', '.')),
        'Percentage': df['Percentage'].apply(lambda x: f'{x:.2f}\\%')
    })
    
    caption = "Distribution of Samples by Class in the EEG Eye State Dataset."
    label = "tab:class_distribution"
    column_format = "lrr" 
    
    # Geramos o LaTeX, mas pedimos para NÃO INSERIR CABEÇALHO (header=False)
    # nem linhas (usando default, que é sem linhas para versões antigas)
    latex_string = df_latex.to_latex(
        index_names=False, # Não exibe o nome do índice (Class, no caso)
        header=False,      # Não exibe a linha de cabeçalho
        column_format=column_format, 
        escape=False       # Mantém o '\\%' como %
    )

    # Pegamos apenas as linhas de DADOS geradas, ignorando \begin{tabular} e \end{tabular}
    linhas_dados = latex_string.splitlines()
    corpo_tabela = "\n".join(linhas_dados[1:-1]).strip()

    # Inserimos o conteúdo gerado em um template limpo com comandos booktabs
    latex_final = f"""
\\begin{{table}}[!htbp]
\\small
\\caption{{{caption}}}
\\centering
\\begin{{center}}
\\begin{{tabular}}{{{column_format}}}
\\toprule
Class & Samples (N) & Percentage (\\%) \\\\
\\midrule
{corpo_tabela}
\\bottomrule
\\end{{tabular}}
\\end{{center}}
\\label{{{label}}}
\\end{{table}}
"""
    return latex_final


# --- GERAÇÃO FINAL DO CÓDIGO LaTeX ---
codigo_latex_final = gerar_latex_analise_classes(df_analise)

print("\n\n#################################################")
print("### CÓDIGO LaTeX DA TABELA DE CLASSES (FINAL) ###")
print("#################################################")
print(codigo_latex_final)



#################################################
### CÓDIGO LaTeX DA TABELA DE CLASSES (FINAL) ###
#################################################

\begin{table}[!htbp]
\small
\caption{Distribution of Samples by Class in the EEG Eye State Dataset.}
\centering
\begin{center}
\begin{tabular}{lrr}
\toprule
Class & Samples (N) & Percentage (\%) \\
\midrule
\toprule
\midrule
Eyes Closed (1) & 6.033 & 52.13\% \\
Eyes Open (0) & 5.539 & 47.87\% \\
\bottomrule
\bottomrule
\end{tabular}
\end{center}
\label{tab:class_distribution}
\end{table}



In [ ]:
base_path = r'/data/eeg_eyestate/EEG_Eye_State.csv'
cluster_names = ['kmeans', 'gmm', 'agglomerative']
target_column = 'eyeDetection'
group_column = 'GROUP'

total_global_rows = 0 

all_results_df = pd.DataFrame()

for cluster_name in cluster_names:
    path = f'{base_path}_{cluster_name}.csv'


    df = pd.read_csv(path, sep=';', decimal=',')
    total_global_rows = len(df)

    df[target_column] = df[target_column].astype(int)
    distribution_table = pd.crosstab(
            df[group_column], 
            df[target_column], 
            margins=True,
            normalize=False
        )
    distribution_table = distribution_table.rename(columns={
            0: 'Target 0', 
            1: 'Target 1', 
        })
    
    distribution_table = distribution_table.reset_index()
    distribution_table.insert(0, 'Algorithm', cluster_name.lower())
    all_results_df = pd.concat([all_results_df, distribution_table], ignore_index=True)



final_table = all_results_df[all_results_df[group_column] != 'Total Geral'].copy()
final_table = final_table[final_table[group_column] != 'All'].copy()

final_table['% 0'] = (final_table['Target 0'] / total_global_rows) * 100
final_table['% 1'] = (final_table['Target 1'] / total_global_rows) * 100

final_table = final_table[['Algorithm', group_column, 'Target 0', '% 0', 'Target 1', '% 1']]
final_table.columns = ['Algorithm', 'Cluster', '0', '% 0', '1', '% 1']

print(final_table.to_string(index=False))
print("\nAnálise de distribuição consolidada concluída.")




In [ ]:
from io import StringIO

def generate_latex_table(df):
    """Gera a string LaTeX no formato hierárquico solicitado, com contagem e porcentagem."""
    
    latex_string = StringIO()
    latex_string.write(r'\begin{table}[h!]' + '\n')
    latex_string.write(r'\centering' + '\n')
    latex_string.write(r'\caption{Distribution of the Target Column by Cluster. }' + '\n')
    latex_string.write(r'\label{tab:cluster_target_distribution}' + '\n')
    
    # Define o número de colunas: Algorithm + Cluster + (Contagem + %) x 2 = 6 Colunas (c c r r r r)
    latex_string.write(r'\begin{tabular}{ccrr rr}' + '\n') 
    latex_string.write(r'\toprule' + '\n')

    # Cabeçalho Principal (Colspan 4 para as colunas de Targets)
    latex_string.write(r'\textbf{Algorithm} & \textbf{Cluster} & \multicolumn{4}{c}{\textbf{' + target_column + r'}}' + r' \\' + '\n')
    
    # Cabeçalho Secundário (Subdivisão de Contagem)
    latex_string.write(r'\cmidrule(lr){3-6}' + '\n')
    # Novo Cabeçalho: 0 (Contagem) | 0 (%) | 1 (Contagem) | 1 (%)
    latex_string.write(r' & & \multicolumn{2}{c}{\textbf{0}} & \multicolumn{2}{c}{\textbf{1}}' + r' \\' + '\n') 
    latex_string.write(r'\cmidrule(lr){3-4}\cmidrule(lr){5-6}' + '\n')
    latex_string.write(r' & & \multicolumn{1}{c}{Count} & \multicolumn{1}{c}{(\%)} & \multicolumn{1}{c}{Count} & \multicolumn{1}{c}{(\%)}' + r' \\' + '\n')
    latex_string.write(r'\midrule' + '\n')
    
    current_algo = None
    
    # Itera sobre o DataFrame
    for index, row in df.iterrows():
        algo = row['Algorithm']
        cluster = str(row['Cluster'])
        
        # Formata Contagens (com ponto como separador de milhar)
        count_0 = f"{row['0']:,}".replace(',', 'X').replace('.', ',').replace('X', '.')
        count_1 = f"{row['1']:,}".replace(',', 'X').replace('.', ',').replace('X', '.')
        
        # Formata Porcentagens (1 casa decimal)
        percent_0 = f"{row['% 0']:.1f}"
        percent_1 = f"{row['% 1']:.1f}"

        # Verifica se o algoritmo mudou
        if algo != current_algo:
            if current_algo is not None:
                latex_string.write(r'\midrule' + '\n')
                
            num_clusters = df[df['Algorithm'] == algo].shape[0]
            
            # Cria a linha do Algoritmo com \multirow
            latex_string.write(r'\multirow{' + str(num_clusters) + r'}{*}{\textbf{' + algo + r'}}')
            current_algo = algo
        else:
            # Coluna vazia para \multirow
            latex_string.write(r'')
            
        # Linhas de Dados do Cluster (Count | % | Count | %)
        latex_string.write(f" & {cluster} & {count_0} & ({percent_0}) & {count_1} & ({percent_1})" + r' \\' + '\n')
        
    latex_string.write(r'\bottomrule' + '\n')
    latex_string.write(r'\end{tabular}' + '\n')
    latex_string.write(r'\end{table}' + '\n')
    
    return latex_string.getvalue()

latex_output = generate_latex_table(final_table)
print(latex_output)

In [ ]:
import pandas as pd
import os
import sys

base_path = r'/data/eeg_eyestate/EEG_Eye_State'
cluster_names = ['kmeans', 'gmm', 'agglomerative']
target_column = 'eyeDetection'
group_column = 'GROUP'
sets = ['train', 'test']

all_results_df = pd.DataFrame()

for cluster_name in cluster_names:
    for data_set in sets:
        path = os.path.join(os.path.dirname(base_path), f'EEG_Eye_State_{cluster_name}_{data_set}.csv')
        
        df = pd.read_csv(path, sep=';', decimal=',')
        total_set_rows = len(df)

        df[target_column] = df[target_column].astype(int)
        
        distribution_table = pd.crosstab(
            df[group_column], 
            df[target_column], 
            margins=True,
            normalize=False
        )
        
        distribution_table_perc = (distribution_table / total_set_rows) * 100
        
        
        counts_df = distribution_table.rename(columns={0: 'Count_0', 1: 'Count_1', 'All': 'Total_Count'}).reset_index()
        perc_df = distribution_table_perc.rename(columns={0: 'Perc_0', 1: 'Perc_1', 'All': 'Total_Perc'}).reset_index()
        
        temp_df = counts_df.merge(perc_df, on=group_column)
        temp_df.insert(0, 'Set', data_set.capitalize())
        temp_df.insert(0, 'Algorithm', cluster_name.lower())
        
        temp_df = temp_df[temp_df[group_column] != 'All'].copy()
        
        all_results_df = pd.concat([all_results_df, temp_df], ignore_index=True)



final_table = all_results_df[[
    'Algorithm', 'Set', group_column, 
    'Count_0', 'Perc_0', 'Count_1', 'Perc_1', 
    'Total_Count'
]].copy()

final_table.columns = [
    'Algoritmo', 'Conjunto', 'Cluster', 
    'Contagem (0)', '% (0)', 'Contagem (1)', '% (1)', 
    'Total'
]

final_table['% (0)'] = final_table['% (0)'].map('{:.2f}%'.format)
final_table['% (1)'] = final_table['% (1)'].map('{:.2f}%'.format)

final_table = final_table.sort_values(by=['Algoritmo', 'Conjunto', 'Cluster'])

final_table_markdown = final_table.to_markdown(index=False, floatfmt=(".0f"))

print("Análise de distribuição consolidada (Treino e Teste) concluída.")
print("\nSugestão de Estrutura da Tabela:\n")
print(final_table_markdown)
